# PDEformer Fine-Tuning Demo: `gray_scott_reaction_diffusion`

This notebook demonstrates the Gray-Scott Well fine-tune run and evaluates the resulting checkpoint. It uses the equation registry in `src.data.well_equations`, so PDEformer receives the dataset-specific Gray-Scott PDE DAG.

The completed run used one selected field (`field_indices=0`) from `gray_scott_reaction_diffusion`, with the successful checkpoint saved under `exp/finetune/the_well/gray_scott_reaction_diffusion/model-L/2026-06-29-14-32-31/`.


## Equation

$A_t=\delta_A\Delta A-AB^2+f(1-A)$
$B_t=\delta_B\Delta B+AB^2-(f+k)B$

The Well metadata supplies normalized trajectories; this adapter builds the registered Gray-Scott DAG and feeds the selected initial-condition field to PDEformer.


In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

workspace_root = Path.cwd().resolve()
if not (workspace_root / "src").exists():
    workspace_root = workspace_root.parent
workspace_root = workspace_root.resolve()
notebook_dir = workspace_root / "notebooks"

# A failed earlier import can leave PyYAML half-initialized in a notebook kernel.
# Clear it before OmegaConf imports yaml internally.
for name in list(sys.modules):
    if name == "yaml" or name.startswith("yaml."):
        del sys.modules[name]

# Keep notebook/workspace paths out while importing PyYAML, then add the repo root back.
blocked_paths = {"", str(Path.cwd().resolve()), str(notebook_dir.resolve())}
sys.path[:] = [path for path in sys.path if path not in blocked_paths]
yaml = importlib.import_module("yaml")
print("yaml module loaded from", yaml.__file__)

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))
os.chdir(workspace_root)

from src.data.well_equations import get_well_equation_spec

WELL_DATASET = "gray_scott_reaction_diffusion"
WELL_BASE_PATH = "hf://datasets/polymathic-ai/"  # or a local /path/to/the_well root
TRAIN_SPLIT = "train"
TEST_SPLIT = "test"
FIELD_INDICES = "0"
MAX_FIELDS = 2

RUN_DIR = Path("eexp/finetune/the_well/gray_scott_reaction_diffusion/model-L/2026-06-29-15-05-27")
FINETUNED_CHECKPOINT = RUN_DIR / "ckpt/model_last.ckpt"
RUN_CONFIG = Path("slurm_logs/the_well_finetune_gray_scott_reaction_diffusion_9268.yaml")
EVAL_OUTPUT = Path("exp/the_well/gray_scott_reaction_diffusion_test_finetuned_eval.json")

spec = get_well_equation_spec(WELL_DATASET)
print(spec.pde_latex)
print("checkpoint:", FINETUNED_CHECKPOINT)
print("checkpoint exists:", FINETUNED_CHECKPOINT.exists())


yaml module loaded from /mnt/data/home/arjun/pdeformer-2/.venv/lib/python3.12/site-packages/yaml/__init__.py
$A_t=\delta_A\Delta A-AB^2+f(1-A)$\n$B_t=\delta_B\Delta B+AB^2-(f+k)B$
checkpoint: eexp/finetune/the_well/gray_scott_reaction_diffusion/model-L/2026-06-29-15-05-27/ckpt/model_best.ckpt
checkpoint exists: False


## Completed Fine-Tune Run

The Slurm run below fine-tuned `model-L.pt` for 5 epochs on the Gray-Scott training split and saved `model_best.ckpt` plus the copied config and metric tables.


In [17]:
summary_path = RUN_DIR / "wandb/run-20260629_143236-o1wgpjof/files/wandb-summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    keys = [
        "train/loss",
        "train_the_well_gray_scott_reaction_diffusion:train/eval_error_mean",
        "train_the_well_gray_scott_reaction_diffusion:train/l2_error_mean",
        "test_the_well_gray_scott_reaction_diffusion:valid/eval_error_mean",
        "test_the_well_gray_scott_reaction_diffusion:valid/l2_error_mean",
    ]
    for key in keys:
        print(f"{key}: {summary.get(key)}")
else:
    print("No W&B summary found at", summary_path)


No W&B summary found at exp/finetune/the_well/gray_scott_reaction_diffusion/model-L/2026-06-29-15-05-27/wandb/run-20260629_143236-o1wgpjof/files/wandb-summary.json


## Local Dataloader Smoke Test

Run this before a new fine-tune or evaluation. It exercises the same Well dataloader path that previously caught the shape/layout issues, without launching model training.


In [11]:
smoke_command = [
    sys.executable,
    "scripts/smoke_the_well_dataloader.py",
    "-c",
    str(RUN_CONFIG),
]
print(" ".join(smoke_command))
subprocess.run(smoke_command, check=True)


/mnt/data/home/arjun/pdeformer-2/.venv/bin/python scripts/smoke_the_well_dataloader.py -c slurm_logs/the_well_finetune_gray_scott_reaction_diffusion_9261.yaml
train the_well gray_scott_reaction_diffusion:train
  [0] (1, 264, 1)
  [1] (1, 256, 1)
  [2] (1, 2, 16384, 5)
  [3] (1, 264)
  [4] (1, 264)
  [5] (1, 264, 264)
  [6] (1, 264, 264)
  [7] (1, 16384, 4)
  [8] (1, 16384, 1)
  [9] (1,)


CompletedProcess(args=['/mnt/data/home/arjun/pdeformer-2/.venv/bin/python', 'scripts/smoke_the_well_dataloader.py', '-c', 'slurm_logs/the_well_finetune_gray_scott_reaction_diffusion_9261.yaml'], returncode=0)

## Reproduce or Extend Fine-Tuning

This is the Slurm command used by the wrapper style. Override `TRAIN_SAMPLES`, `TEST_SAMPLES`, `EPOCHS`, or `FIELD_INDICES` if you want a larger or different run.


In [8]:
slurm_command = f'''sbatch --export=ALL,WELL_BASE_PATH={WELL_BASE_PATH},WELL_DATASET={WELL_DATASET},TRAIN_SPLIT={TRAIN_SPLIT},TEST_SPLIT={TEST_SPLIT},CHECKPOINT=model-L.pt,FIELD_INDICES={FIELD_INDICES},MAX_FIELDS={MAX_FIELDS} scripts/submit_the_well_finetune_slurm.sh'''
print(slurm_command)


sbatch --export=ALL,WELL_BASE_PATH=hf://datasets/polymathic-ai/,WELL_DATASET=gray_scott_reaction_diffusion,TRAIN_SPLIT=train,TEST_SPLIT=valid,CHECKPOINT=model-L.pt,FIELD_INDICES=0,MAX_FIELDS=2 scripts/submit_the_well_finetune_slurm.sh


## Evaluate the Fine-Tuned Checkpoint

This evaluates `model_best.ckpt` with the same equation-aware preset. It writes a JSON report containing per-sample examples, selected channel metadata, RMSE-style PDEformer metrics, and The Well validation metric suite when available.


In [12]:
eval_command = [
    sys.executable,
    "scripts/evaluate_the_well.py",
    "--config", "configs/inference/model-L.yaml",
    "--checkpoint", str(FINETUNED_CHECKPOINT),
    "--well-base-path", WELL_BASE_PATH,
    "--well-dataset", WELL_DATASET,
    "--split", TEST_SPLIT,
    "--num-samples", "4",
    "--field-indices", FIELD_INDICES,
    "--max-fields", str(MAX_FIELDS),
    "--pde-preset", "well_equation",
    "--well-normalization", "zscore",
    "--device-target", "GPU",
    "--output", str(EVAL_OUTPUT),
]
print(" ".join(eval_command))
subprocess.run(eval_command, check=True)


/mnt/data/home/arjun/pdeformer-2/.venv/bin/python scripts/evaluate_the_well.py --config configs/inference/model-L.yaml --checkpoint exp/finetune/the_well/gray_scott_reaction_diffusion/model-L/2026-06-29-14-32-31/ckpt/model_best.ckpt --well-base-path hf://datasets/polymathic-ai/ --well-dataset gray_scott_reaction_diffusion --split test --num-samples 4 --field-indices 0 --max-fields 2 --pde-preset well_equation --well-normalization zscore --device-target GPU --output exp/the_well/gray_scott_reaction_diffusion_valid_finetuned_eval.json


/mnt/data/home/arjun/pdeformer-2/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


{
  "well_dataset": "gray_scott_reaction_diffusion",
  "split": "test",
  "num_samples_requested": 4,
  "num_samples_evaluated": 4,
  "pde_preset": "well_equation",
  "selected_channel_indices": [
    0
  ],
  "well_channel_names_by_tensor_order": {
    "0": [
      "A",
      "B"
    ],
    "1": [],
    "2": []
  },
  "normalization": {
    "coordinates_to_unit_box": true,
    "time_to_unit_interval": true,
    "well_input_normalization": "zscore",
    "normalization_path": "hf://datasets/polymathic-ai/gray_scott_reaction_diffusion/stats.yaml",
    "predictions_denormalized_for_metrics": true
  },
  "metrics": {
    "relative_l2": {
      "mean": 0.13923721946775913,
      "median": 0.12469589337706566,
      "max": 0.19760791957378387
    },
    "relative_l2_by_channel": {
      "0": {
        "mean": 0.13923721946775913,
        "median": 0.12469589337706566,
        "max": 0.19760791957378387
      }
    },
    "well_metrics": {
      "RMSE": {
        "mean": 0.13176796957850456,


CompletedProcess(args=['/mnt/data/home/arjun/pdeformer-2/.venv/bin/python', 'scripts/evaluate_the_well.py', '--config', 'configs/inference/model-L.yaml', '--checkpoint', 'exp/finetune/the_well/gray_scott_reaction_diffusion/model-L/2026-06-29-14-32-31/ckpt/model_best.ckpt', '--well-base-path', 'hf://datasets/polymathic-ai/', '--well-dataset', 'gray_scott_reaction_diffusion', '--split', 'test', '--num-samples', '4', '--field-indices', '0', '--max-fields', '2', '--pde-preset', 'well_equation', '--well-normalization', 'zscore', '--device-target', 'GPU', '--output', 'exp/the_well/gray_scott_reaction_diffusion_valid_finetuned_eval.json'], returncode=0)

## Read the Evaluation Report

After running the evaluation cell, use this cell to inspect the aggregate metrics and the first example's tensor shapes. If you run on a CPU-only machine, change `--device-target GPU` to `CPU` in the previous cell.


In [13]:
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None


def flatten_metrics(metrics, prefix=""):
    rows = []
    for key, value in metrics.items():
        name = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, dict):
            rows.extend(flatten_metrics(value, name))
        else:
            rows.append({"metric": name, "value": value})
    return rows


def format_value(value):
    if isinstance(value, float):
        return f"{value:.6g}"
    return value


def show_table(rows, columns=None):
    if pd is None:
        for row in rows:
            print(row)
        return
    df = pd.DataFrame(rows, columns=columns)
    for column in df.columns:
        df[column] = df[column].map(format_value)
    display(df.style.hide(axis="index"))

if EVAL_OUTPUT.exists():
    result = json.loads(EVAL_OUTPUT.read_text())

    summary_rows = [
        {"field": "dataset", "value": result["well_dataset"]},
        {"field": "split", "value": result["split"]},
        {"field": "samples evaluated", "value": result["num_samples_evaluated"]},
        {"field": "PDE preset", "value": result["pde_preset"]},
        {"field": "selected channels", "value": ", ".join(map(str, result["selected_channel_indices"]))},
        {"field": "normalization", "value": result["normalization"]["well_input_normalization"]},
    ]
    print("Evaluation summary")
    show_table(summary_rows, columns=["field", "value"])

    metric_rows = flatten_metrics(result["metrics"])
    print("Metrics")
    show_table(metric_rows, columns=["metric", "value"])

    if result.get("examples"):
        example_rows = []
        for example in result["examples"]:
            example_rows.append({
                "sample": example.get("sample_index"),
                "input_fields": example.get("input_fields_shape"),
                "output_fields": example.get("output_fields_shape"),
                "space_grid": example.get("space_grid_shape"),
            })
        print("Examples")
        show_table(example_rows, columns=["sample", "input_fields", "output_fields", "space_grid"])
else:
    print("Run the evaluation cell first; missing", EVAL_OUTPUT)


Evaluation summary


field,value
dataset,gray_scott_reaction_diffusion
split,test
samples evaluated,4
PDE preset,well_equation
selected channels,0
normalization,zscore


Metrics


metric,value
relative_l2.mean,0.139237
relative_l2.median,0.124696
relative_l2.max,0.197608
relative_l2_by_channel.0.mean,0.139237
relative_l2_by_channel.0.median,0.124696
relative_l2_by_channel.0.max,0.197608
well_metrics.RMSE.mean,0.131768
well_metrics.RMSE.median,0.118839
well_metrics.RMSE.max,0.18371
well_metrics.NRMSE.mean,0.139237


Examples


sample,input_fields,output_fields,space_grid
0,"[1, 128, 128, 2]","[1, 128, 128, 2]","[128, 128, 2]"
1,"[1, 128, 128, 2]","[1, 128, 128, 2]","[128, 128, 2]"
2,"[1, 128, 128, 2]","[1, 128, 128, 2]","[128, 128, 2]"
